# RSI / price-change divergence — flipped to LONG (BTC 1H, walk-forward + W4 deep-dive)

**Status** — Both SHORT and LONG variants of the divergence + filter strategy fail walk-forward across W1-W3 (2018-2024). Only W4 (2024-2026) is profitable. This notebook now investigates *why* — what makes W4 different — so the insight can feed a future strategy even if this one is parked.

**Trade rules (LONG)** — bullish-divergence at RSI<30, TP at RSI≥70, stop after RSI rises above 30 if it returns to ≤30. No time stop. See sim-core cell for the unified short/long simulator.

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import date, datetime, timezone
import numpy as np

_p = Path.cwd()
while _p != _p.parent and not (_p / 'jplus').is_dir():
    _p = _p.parent
REPO_ROOT = _p
TRADER_DB = REPO_ROOT / 'data' / 'trader.db'

START_D = date(2024, 5, 9)
END_D   = date(2026, 5, 8)

def _ts(d): return int(datetime(d.year, d.month, d.day, tzinfo=timezone.utc).timestamp())
START_TS = _ts(START_D)
END_TS   = _ts(END_D) + 86400

WF_WINDOWS = [
    ('W1 2018-05→2020-05', date(2018, 5, 9), date(2020, 5, 9)),
    ('W2 2020-05→2022-05', date(2020, 5, 9), date(2022, 5, 9)),
    ('W3 2022-05→2024-05', date(2022, 5, 9), date(2024, 5, 9)),
    ('W4 2024-05→2026-05', date(2024, 5, 9), date(2026, 5, 8)),
]

FEE_RT     = 0.0010
LOWER_RSI  = 30.0
UPPER_RSI  = 70.0

print(f'Reference window: {START_D} → {END_D}')
print(f'WF windows: {len(WF_WINDOWS)}')

In [ ]:
def hourly_btc_full(con):
    rows = con.execute('''
        SELECT timestamp, close, volume, volume_buy, volume_sell
        FROM cd_spot_binance ORDER BY timestamp
    ''').fetchall()
    n = len(rows)
    ts       = np.empty(n, dtype=np.int64)
    close    = np.empty(n, dtype=float)
    vol      = np.empty(n, dtype=float)
    vol_buy  = np.empty(n, dtype=float)
    vol_sell = np.empty(n, dtype=float)
    for i, r in enumerate(rows):
        ts[i]       = r[0]
        close[i]    = r[1]
        vol[i]      = r[2]
        vol_buy[i]  = r[3] if r[3] is not None else np.nan
        vol_sell[i] = r[4] if r[4] is not None else np.nan
    return ts, close, vol, vol_buy, vol_sell

con = sqlite3.connect(str(TRADER_DB))
FULL_TS, FULL_CLOSE, FULL_VOL, FULL_VBUY, FULL_VSELL = hourly_btc_full(con)
con.close()
print(f'Full 1H history: {len(FULL_TS):,} bars')

In [ ]:
def wilder_rsi(close, period=14):
    delta = np.diff(close, prepend=close[0])
    gain  = np.where(delta > 0,  delta, 0.0)
    loss  = np.where(delta < 0, -delta, 0.0)
    avg_g = np.full_like(close, np.nan, dtype=float)
    avg_l = np.full_like(close, np.nan, dtype=float)
    avg_g[period] = gain[1:period+1].mean()
    avg_l[period] = loss[1:period+1].mean()
    for i in range(period+1, len(close)):
        avg_g[i] = (avg_g[i-1] * (period-1) + gain[i]) / period
        avg_l[i] = (avg_l[i-1] * (period-1) + loss[i]) / period
    rs = avg_g / np.where(avg_l == 0, np.nan, avg_l)
    return 100 - 100 / (1 + rs)

def rolling_z(x, window):
    z = np.full_like(x, np.nan, dtype=float)
    for i in range(window, len(x)):
        w = x[i-window:i]
        m, s = w.mean(), w.std(ddof=0)
        if s > 0:
            z[i] = (x[i] - m) / s
    return z

def features(close, volume, vol_buy, vol_sell, *,
             rsi_period=14, z_window=48, vol_z_window=48):
    log_ret = np.diff(np.log(close), prepend=np.log(close[0]))
    log_vol = np.log(np.maximum(volume, 1e-9))
    rsi     = wilder_rsi(close, rsi_period)
    pz      = rolling_z(log_ret, z_window)
    vol_z   = rolling_z(log_vol, vol_z_window)
    total   = vol_buy + vol_sell
    imb     = np.where(total > 0, (vol_buy - vol_sell) / np.where(total > 0, total, 1.0), np.nan)
    return log_ret, rsi, pz, vol_z, imb

FULL_LOGRET, FULL_RSI, FULL_PZ, FULL_VOLZ, FULL_IMB = features(
    FULL_CLOSE, FULL_VOL, FULL_VBUY, FULL_VSELL, z_window=48, vol_z_window=48)

print(f'Full RSI tail: {np.round(FULL_RSI[-5:], 1)}')

In [ ]:
def divergence_flags(rsi, pz, *, mode='long', n=5, theta_r=5.0, theta_p=0.0,
                     entry_rsi_min=70.0, entry_rsi_max=30.0,
                     vol_z=None, vol_burst_min=None,
                     imb=None, imb_max=None, imb_min=None):
    rsi_d = np.full_like(rsi, np.nan); rsi_d[n:] = rsi[n:] - rsi[:-n]
    pz_d  = np.full_like(pz,  np.nan); pz_d[n:]  = pz[n:]  - pz[:-n]
    valid = ~np.isnan(rsi_d) & ~np.isnan(pz_d) & ~np.isnan(rsi)
    if mode == 'short':
        flag = (rsi_d > theta_r) & (pz_d < theta_p) & (rsi > entry_rsi_min) & valid
        if imb is not None and imb_max is not None:
            flag = flag & (imb < imb_max) & ~np.isnan(imb)
    else:
        flag = (rsi_d < -theta_r) & (pz_d > theta_p) & (rsi < entry_rsi_max) & valid
        if imb is not None and imb_min is not None:
            flag = flag & (imb > imb_min) & ~np.isnan(imb)
    if vol_z is not None and vol_burst_min is not None:
        flag = flag & (vol_z > vol_burst_min) & ~np.isnan(vol_z)
    return flag

def _close_trade(pos, close, i, reason, mode):
    win = close[pos['entry_i']:i+1]
    if mode == 'short':
        ret = (pos['entry_px'] - close[i]) / pos['entry_px']
        mae = -(win.max() - pos['entry_px']) / pos['entry_px']
        mfe =  (pos['entry_px'] - win.min()) / pos['entry_px']
    else:
        ret = (close[i] - pos['entry_px']) / pos['entry_px']
        mae = (win.min() - pos['entry_px']) / pos['entry_px']
        mfe = (win.max() - pos['entry_px']) / pos['entry_px']
    return {
        'entry_i': pos['entry_i'], 'exit_i': i,
        'entry_px': pos['entry_px'], 'exit_px': close[i],
        'hold': i - pos['entry_i'],
        'ret': ret, 'mae': mae, 'mfe': mfe, 'reason': reason,
    }

def simulate(close, rsi, flag, *, mode='long',
             lower_rsi=LOWER_RSI, upper_rsi=UPPER_RSI):
    trades = []
    pos = None
    for i in range(len(close)):
        if pos is None:
            if flag[i]:
                pos = {'entry_i': i, 'entry_px': close[i], 'armed': False}
        else:
            r = rsi[i]
            if not np.isnan(r):
                if mode == 'short':
                    if r <= lower_rsi:
                        trades.append(_close_trade(pos, close, i, 'tp', 'short')); pos = None; continue
                    if not pos['armed']:
                        if r < upper_rsi: pos['armed'] = True
                    elif r >= upper_rsi:
                        trades.append(_close_trade(pos, close, i, 'stop', 'short')); pos = None
                else:
                    if r >= upper_rsi:
                        trades.append(_close_trade(pos, close, i, 'tp', 'long')); pos = None; continue
                    if not pos['armed']:
                        if r > lower_rsi: pos['armed'] = True
                    elif r <= lower_rsi:
                        trades.append(_close_trade(pos, close, i, 'stop', 'long')); pos = None
    if pos is not None:
        trades.append(_close_trade(pos, close, len(close)-1, 'end', mode))
    return trades

def levered_returns(trades, lev, fee_rt=FEE_RT):
    liq_thresh = -1.0 / lev
    out = np.empty(len(trades), dtype=float)
    liq = np.zeros(len(trades), dtype=bool)
    for k, t in enumerate(trades):
        if t['mae'] <= liq_thresh:
            out[k] = -1.0; liq[k] = True
        else:
            out[k] = lev * t['ret'] - lev * fee_rt
    return out, liq

def window_mask(ts, d_start, d_end):
    return (ts >= _ts(d_start)) & (ts < _ts(d_end) + 86400)

In [ ]:
# ---------- W4 deep-dive: per-window LONG trade anatomy ----------
# Goal: identify what numerically separates W4 (the only profitable window)
# from W1-W3 (all losing). Same params, no filter, 10x leverage.

def enrich(t, rsi):
    win_rsi = rsi[t['entry_i']:t['exit_i']+1]
    return {**t,
            'entry_rsi': float(rsi[t['entry_i']]),
            'max_rsi':   float(np.nanmax(win_rsi))}

PARAMS = dict(n=5, theta_r=5.0, theta_p=0.0)

windows_data = {}
for lab, d0, d1 in WF_WINDOWS:
    m = window_mask(FULL_TS, d0, d1)
    flag = divergence_flags(FULL_RSI[m], FULL_PZ[m], mode='long',
                            entry_rsi_max=30.0, **PARAMS)
    trades = simulate(FULL_CLOSE[m], FULL_RSI[m], flag, mode='long')
    enriched = [enrich(t, FULL_RSI[m]) for t in trades]
    rets, liq = levered_returns(trades, 10)
    windows_data[lab] = (enriched, rets, liq, m)

print('=== Per-window LONG anatomy — n=5, θ_r=5, θ_p=0, no filter, lev=10x ===')
print(f'{"window":<22} {"N":>4} {"TP":>3} {"STP":>4} {"liq":>4} {"win%":>6} '
      f'{"avg_TP%":>8} {"avg_STP%":>9} {"avg_raw_TP%":>11} {"avg_raw_STP%":>12} '
      f'{"med_h":>6} {"med_max_rsi_stop":>17} {"med_entry_rsi":>14}')

for lab, _, _ in [(l, d0, d1) for l, d0, d1 in WF_WINDOWS]:
    enriched, rets, liq, _ = windows_data[lab]
    if not enriched:
        print(f'{lab:<22} 0 trades'); continue
    n = len(enriched)
    tps  = [(t, r) for t, r in zip(enriched, rets) if t['reason'] == 'tp']
    stps = [(t, r) for t, r in zip(enriched, rets) if t['reason'] == 'stop']
    avg_tp     = np.mean([r for _, r in tps])  * 100  if tps  else float('nan')
    avg_st     = np.mean([r for _, r in stps]) * 100  if stps else float('nan')
    avg_raw_tp = np.mean([t['ret'] for t, _ in tps])  * 100  if tps  else float('nan')
    avg_raw_st = np.mean([t['ret'] for t, _ in stps]) * 100  if stps else float('nan')
    med_max_stop = np.median([t['max_rsi'] for t, _ in stps]) if stps else float('nan')
    med_entry    = np.median([t['entry_rsi'] for t in enriched])
    holds        = np.array([t['hold'] for t in enriched])
    print(f'{lab:<22} {n:>4} {len(tps):>3} {len(stps):>4} {int(liq.sum()):>4} '
          f'{(rets>0).mean()*100:>5.1f}% {avg_tp:>+7.2f}% {avg_st:>+8.2f}% '
          f'{avg_raw_tp:>+10.3f}% {avg_raw_st:>+11.3f}% '
          f'{int(np.median(holds)):>5}h {med_max_stop:>16.1f} {med_entry:>13.1f}')

In [ ]:
# ---------- W4 winners vs W1 losers ----------

def list_trades(window_label, trades, ts_window, n=10, mode='best'):
    rsi_pad = '  '
    print(f'\n--- {window_label} {"top" if mode=="best" else "bottom"} {n} trades by raw return ---')
    indexed = sorted(trades, key=lambda t: -t['ret'] if mode == 'best' else t['ret'])
    print(f'{rsi_pad}{"#":>3}  {"entry_dt":<17}  {"hold":>5}  '
          f'{"entry_rsi":>9}  {"max_rsi":>7}  {"raw_ret":>8}  {"reason":>6}')
    for k, t in enumerate(indexed[:n]):
        entry_dt = datetime.fromtimestamp(int(ts_window[t['entry_i']]), tz=timezone.utc)
        print(f'{rsi_pad}{k+1:>3}  {entry_dt.strftime("%Y-%m-%d %H:%M")}  {t["hold"]:>4}h  '
              f'{t["entry_rsi"]:>8.1f}  {t["max_rsi"]:>7.1f}  '
              f'{t["ret"]*100:>+7.2f}%  {t["reason"]:>6}')

for lab in ['W4 2024-05→2026-05', 'W1 2018-05→2020-05']:
    enriched, rets, liq, m = windows_data[lab]
    ts_window = FULL_TS[m]
    list_trades(lab, enriched, ts_window, n=10, mode='best')

print()
for lab in ['W1 2018-05→2020-05', 'W4 2024-05→2026-05']:
    enriched, rets, liq, m = windows_data[lab]
    ts_window = FULL_TS[m]
    list_trades(lab, enriched, ts_window, n=10, mode='worst')

In [ ]:
# ---------- RSI behavior per window (independent of strategy) ----------
# Maybe RSI just behaves differently across regimes. If RSI < 30 means something
# different in W1 vs W4, that explains the strategy's regime dependence.

print('=== RSI distribution per window (the underlying signal source) ===')
print(f'{"window":<22} {"% RSI<30":>10} {"% RSI<25":>10} {"% RSI<20":>10} '
      f'{"% RSI>70":>10} {"% RSI>75":>10} {"% RSI>80":>10}')
for lab, d0, d1 in WF_WINDOWS:
    m = window_mask(FULL_TS, d0, d1)
    rsi = FULL_RSI[m]
    valid = ~np.isnan(rsi)
    rsi = rsi[valid]
    print(f'{lab:<22} '
          f'{(rsi<30).mean()*100:>9.2f}% {(rsi<25).mean()*100:>9.2f}% {(rsi<20).mean()*100:>9.2f}% '
          f'{(rsi>70).mean()*100:>9.2f}% {(rsi>75).mean()*100:>9.2f}% {(rsi>80).mean()*100:>9.2f}%')

# How often does RSI cross from <30 to >=70 within X bars (TP-eligibility test)?
print(f'\n=== Conditional: given RSI dipped below 30, % of crossings that subsequently reach RSI>=70 within K bars ===')
print(f'{"window":<22} {"crossings":>10} {"TP within 24h":>14} {"within 48h":>11} {"within 168h":>12} {"within 500h":>12}')
for lab, d0, d1 in WF_WINDOWS:
    m = window_mask(FULL_TS, d0, d1)
    rsi = FULL_RSI[m]
    crossings = []
    in_oversold = False
    for i in range(len(rsi)):
        if np.isnan(rsi[i]):
            continue
        if not in_oversold and rsi[i] < 30:
            in_oversold = True
            crossings.append(i)
        elif in_oversold and rsi[i] >= 30:
            in_oversold = False
    n_cross = len(crossings)
    counts = {h: 0 for h in (24, 48, 168, 500)}
    for i in crossings:
        for h in counts:
            window_rsi = rsi[i:min(i+h+1, len(rsi))]
            if np.any(window_rsi >= 70):
                counts[h] += 1
    print(f'{lab:<22} {n_cross:>10} '
          f'{counts[24]/max(n_cross,1)*100:>13.1f}% {counts[48]/max(n_cross,1)*100:>10.1f}% '
          f'{counts[168]/max(n_cross,1)*100:>11.1f}% {counts[500]/max(n_cross,1)*100:>11.1f}%')

## Reading the deep-dive

Three diagnostic angles, all looking for what makes W4 different:

1. **Trade anatomy table**
   - `avg_raw_TP%` — when a TP fires, how big is the price move on average. If W4's TP wins are bigger (deeper bounce from oversold), that's part of the answer.
   - `avg_raw_STP%` — how bad are stop-outs on raw price. If W1's stops are catastrophic vs W4's, regime tells you.
   - `med_max_rsi_stop` — for stops, the highest RSI reached during hold. **This is the key**: if W4 stops typically reached RSI≈55 (almost made it) while W1 stops only reached RSI≈40 (never got going), W4 was *almost* a winning environment.

2. **Top-10 winners and losers** — eyeball-readable trade list for W4 and W1. Look at entry dates and RSI levels — were W4's winners clustered around specific events (post-FOMC, ETF flow news)? Were W1's losers all in the 2018 winter or the covid crash?

3. **RSI distribution per window** (independent of strategy)
   - If RSI<30 fires much less often in W4 than in W1, the entries may have been more selective by chance.
   - The crossings-to-TP table is the cleanest test: of all times RSI dipped below 30, what fraction subsequently reached RSI≥70 within 24h / 48h / 7d / ~3w? **If W4's RSI<30 dips reach RSI≥70 much more often than W1's, the regime literally pre-disposes more TPs**, and we don't need a 'strategy edge' explanation.

If finding (3) dominates, the takeaway for any future strategy is: **detect the regime first** (e.g. by measuring TP-rate of a synthetic RSI-cross strategy on a rolling basis), then enable mean-reversion only when the regime supports it. That's a deeper insight than tweaking divergence parameters.